# Pose Detection in a Notebook: Toggle **MoveNet** ↔ **BlazePose**

This notebook lets you:
- Capture frames from your **webcam** (or a video file).
- Run **either** MoveNet (TensorFlow Hub) **or** BlazePose (MediaPipe Tasks).
- Visualize keypoints and a simple **movement** metric.
- (Optional) Compute a joint angle (knee) and count reps.

> **How to use**: set `MODEL` to `"movenet"` or `"blazepose"` in the **Configuration** cell and run cells top‑to‑bottom.

## Why two backends?
- **MoveNet** (TF Hub / TFLite): 17 COCO keypoints. Fast and lightweight. Great for body-only keypoints.
- **BlazePose** (MediaPipe Tasks): 33 keypoints including hands/feet/face-ish. Nice temporal smoothing in `VIDEO` mode.

Both consume the **same webcam frames**; only the detector differs.

## 1) Install (pick what you need)

> You can install **both** stacks, or just the one you plan to use. Restart kernel after installing if needed.

```bash
# --- MoveNet (TensorFlow Hub) ---
pip install "tensorflow>=2.14,<2.18" tensorflow-hub opencv-python numpy

# --- BlazePose (MediaPipe) ---
pip install mediapipe==0.10.14 opencv-python numpy
```

In [6]:
# 2) Imports
# Note: Import errors are okay if you're only using the other backend.
import os, sys, cv2, numpy as np
from collections import deque

In [7]:
# 3) Configuration
MODEL = "blazepose"      # choose: "movenet" or "blazepose"
SOURCE = 0             # 0 = default webcam; or path string like "video.mp4"
DRAW_SKELETON = True   # draw connecting lines
SCORE_THRESH = 0.5     # keypoint score threshold
print(f"Backend selected: {MODEL}, source: {SOURCE}")

Backend selected: blazepose, source: 0


## 4) Unified wrapper: `PoseDetector`

This class hides backend differences so downstream logic is identical:

- `infer(frame_bgr) -> list[(x, y, score)]` in **pixel** coordinates
- `draw(frame_bgr, pts, score_thresh)` overlays keypoints & skeleton
- Internally routes to:
  - MoveNet via **TensorFlow Hub** (Lightning 192×192 by default)
  - BlazePose via **MediaPipe Tasks** (`VIDEO` mode for temporal smoothing)

In [8]:
class PoseDetector:
    def __init__(self, backend="movenet"):
        backend = backend.lower()
        self.backend = backend
        if backend == "movenet":
            self._init_movenet()
        elif backend == "blazepose":
            self._init_blazepose()
        else:
            raise ValueError("backend must be 'movenet' or 'blazepose'")

    # ---------- MoveNet (TF Hub) ----------
    def _init_movenet(self):
        try:
            import tensorflow as tf
            import tensorflow_hub as hub
        except Exception as e:
            raise RuntimeError(
                "MoveNet requires tensorflow + tensorflow-hub. "
                "Install: pip install 'tensorflow>=2.14,<2.18' tensorflow-hub"
            ) from e

        self.tf = tf
        self.hub = hub
        self.input_size = 192  # 192 for Lightning; 256 for Thunder
        self.model = self.hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4")

        # COCO skeleton edges (indices for 17 keypoints)
        self._edges = [
            (5,7),(7,9), (6,8),(8,10), (5,6),
            (5,11),(6,12), (11,12), (11,13),(13,15), (12,14),(14,16),
            (0,1),(0,2),(1,3),(2,4)
        ]

    def _infer_movenet(self, frame_bgr):
        h, w = frame_bgr.shape[:2]
        img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        tf = self.tf
        img_tf = tf.convert_to_tensor(img_rgb)
        img_tf = tf.image.resize_with_pad(img_tf, self.input_size, self.input_size)
        input_img = tf.cast(img_tf, dtype=tf.int32)[tf.newaxis, ...]  # (1,H,W,3)
        out = self.model.signatures["serving_default"](input_img)["output_0"].numpy()  # (1,1,17,3)
        kps = out[0,0,:,:]  # (17, 3) -> (y, x, score) normalized
        pts = []
        for (yy, xx, sc) in kps:
            x = float(xx) * w
            y = float(yy) * h
            pts.append((x, y, float(sc)))
        return pts  # list of (x,y,score)

    # ---------- BlazePose (MediaPipe Tasks) ----------
    def _init_blazepose(self):
        try:
            import mediapipe as mp
            from mediapipe.tasks import python as mp_tasks
            from mediapipe.tasks.python import vision
        except Exception as e:
            raise RuntimeError(
                "BlazePose requires mediapipe==0.10.14. Install: pip install mediapipe==0.10.14"
            ) from e
        self.mp = mp
        self.mp_tasks = mp_tasks
        self.vision = vision

        base_options = self.mp_tasks.BaseOptions(
            model_asset_path=self.vision.PoseLandmarkerModel.POSE_LANDMARKER_FULL
        )
        self.options = self.vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=self.vision.RunningMode.VIDEO,
            num_poses=1,
            min_pose_detection_confidence=0.5,
            min_pose_tracking_confidence=0.5,
            min_pose_presence_confidence=0.5,
            output_segmentation_masks=False
        )
        self.detector = self.vision.PoseLandmarker.create_from_options(self.options)
        self._edges = list(self.mp.solutions.pose.POSE_CONNECTIONS)
        self._timestamp_ms = 0

    def _infer_blazepose(self, frame_bgr):
        h, w = frame_bgr.shape[:2]
        img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        mp_image = self.mp.Image(image_format=self.mp.ImageFormat.SRGB, data=img_rgb)
        res = self.detector.detect_for_video(mp_image, self._timestamp_ms)
        self._timestamp_ms += 33  # ~30 fps

        if not res.pose_landmarks:
            return []
        lms = res.pose_landmarks[0]
        pts = []
        for lm in lms:
            x = float(lm.x) * w
            y = float(lm.y) * h
            score = float(getattr(lm, "visibility", 0.9))
            pts.append((x, y, score))
        return pts

    # ---------- Unified API ----------
    def infer(self, frame_bgr):
        if self.backend == "movenet":
            return self._infer_movenet(frame_bgr)
        else:
            return self._infer_blazepose(frame_bgr)

    def draw(self, frame_bgr, pts, score_thresh=0.5):
        if not pts:
            return frame_bgr
        # points
        for (x, y, sc) in pts:
            if sc >= score_thresh:
                cv2.circle(frame_bgr, (int(x), int(y)), 3, (0,255,0), -1)
        # edges
        if self._edges:
            for a, b in self._edges:
                if a < len(pts) and b < len(pts):
                    if pts[a][2] >= score_thresh and pts[b][2] >= score_thresh:
                        ax, ay, _ = pts[a]; bx, by, _ = pts[b]
                        cv2.line(frame_bgr, (int(ax), int(ay)), (int(bx), int(by)), (0,255,0), 1)
        return frame_bgr

## 5) Utilities: movement & angles

- `movement_magnitude(prev_pts, curr_pts)`: mean per-keypoint displacement between frames.
- `torso_scale(pts)`: scale factor (mid-shoulders to mid-hips) to normalize movement by person size/distance.
- `angle_3pts(a,b,c)`: angle at **b** in degrees.

In [9]:
def movement_magnitude(prev_pts, curr_pts):
    if not prev_pts or not curr_pts:
        return 0.0
    n = min(len(prev_pts), len(curr_pts))
    if n == 0: return 0.0
    disps = []
    for i in range(n):
        x1,y1,_ = prev_pts[i]
        x2,y2,_ = curr_pts[i]
        disps.append(((x2-x1)**2 + (y2-y1)**2) ** 0.5)
    return float(np.mean(disps)) if disps else 0.0

def angle_3pts(a, b, c):
    a = np.array(a[:2], float); b = np.array(b[:2], float); c = np.array(c[:2], float)
    ba, bc = a-b, c-b
    denom = (np.linalg.norm(ba)*np.linalg.norm(bc) + 1e-6)
    cosang = np.clip(np.dot(ba, bc) / denom, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosang)))

def torso_scale(pts):
    try:
        # MediaPipe indices: shoulders 11/12, hips 23/24
        # MoveNet indices:   shoulders 5/6,  hips 11/12
        if len(pts) >= 25:
            sL, sR, hL, hR = pts[11], pts[12], pts[23], pts[24]
        else:
            sL, sR, hL, hR = pts[5], pts[6], pts[11], pts[12]
        s_mid = ((sL[0]+sR[0])/2.0, (sL[1]+sR[1])/2.0)
        h_mid = ((hL[0]+hR[0])/2.0, (hL[1]+hR[1])/2.0)
        d = ((s_mid[0]-h_mid[0])**2 + (s_mid[1]-h_mid[1])**2)**0.5
        return max(d, 1.0)
    except:
        return 1.0

## 6) Live loop (webcam or video file)

- Press **ESC** to quit.
- Shows:
  - keypoints + skeleton
  - smoothed normalized **movement**
  - **knee angle** and simple **rep counter** (left knee)

In [10]:
def run_demo(model_name, source=0):
    print(f"Starting with backend={model_name}, source={source}")
    detector = PoseDetector(model_name)

    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print("Could not open video source. Try SOURCE=0 for webcam or a valid file path.")
        return

    prev_pts = None
    mov_hist = deque(maxlen=30)
    LOW, HIGH = 70, 160  # squat-like thresholds
    down = False
    reps = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        pts = detector.infer(frame)

        move = movement_magnitude(prev_pts, pts)
        scale = torso_scale(pts) if pts else 1.0
        move_norm = move / scale
        mov_hist.append(move_norm)
        smooth = float(np.mean(mov_hist)) if mov_hist else 0.0
        prev_pts = pts

        # Left knee angle
        knee_angle = None
        if pts:
            if len(pts) >= 28:  # BlazePose indices
                hip, knee, ankle = pts[23], pts[25], pts[27]
            else:               # MoveNet indices
                hip, knee, ankle = pts[11], pts[13], pts[15]
            knee_angle = angle_3pts(hip, knee, ankle)

            if knee_angle < LOW:
                down = True
            if down and knee_angle is not None and knee_angle > HIGH:
                reps += 1
                down = False

        vis = detector.draw(frame.copy(), pts, score_thresh=SCORE_THRESH)
        cv2.putText(vis, f"Movement (norm): {smooth:.2f}", (12, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        if knee_angle is not None:
            cv2.putText(vis, f"Knee angle: {knee_angle:.1f} deg  Reps: {reps}",
                        (12, 56), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

        cv2.imshow("Pose (ESC to quit)", vis)
        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

# Run it (uses the CONFIG cell values)
run_demo(MODEL, SOURCE)

Starting with backend=blazepose, source=0


RuntimeError: BlazePose requires mediapipe==0.10.14. Install: pip install mediapipe==0.10.14

## 7) Tips & troubleshooting

- **No webcam feed?**
  - Ensure a camera is available and not used by another app.
  - Try `SOURCE = 1` (or 2, ...) if you have multiple cameras.
  - Use a file path instead of a camera: `SOURCE = "path/to/video.mp4"`.

- **MoveNet import error**: install TF + TF Hub in the environment and restart kernel.
- **BlazePose import error**: install `mediapipe==0.10.14` and restart kernel.
- **Performance**:
  - MoveNet Lightning (192×192) is faster; swap to **Thunder** by changing the TF Hub URL and `self.input_size = 256`.
  - BlazePose has `model_asset_path` options (`LITE`, `FULL`, `HEAVY`) in MediaPipe if desired.
- **Normalize movement**: we divide by a torso scale (shoulder–hip distance) to reduce camera-distance effects.